In [3]:
import os
import re
import numpy as np
import cv2
import torch
import random


def manual_train_test_split(*arrays, test_size=0.25, random_state=42):
    if random_state is not None:
        np.random.seed(random_state)
        random.seed(random_state)

    n_samples = len(arrays[0])
    indices = np.arange(n_samples)
    np.random.shuffle(indices)

    split_idx = int(n_samples * (1 - test_size))
    train_idx = indices[:split_idx]
    test_idx = indices[split_idx:]

    result = []
    for arr in arrays:
        if isinstance(arr, np.ndarray):
            # Для numpy-массивов работает прямое индексирование
            result.append(arr[train_idx])
            result.append(arr[test_idx])
        else:
            # Для списков используем генератор + конвертируем индексы в list
            train_list = train_idx.tolist()
            test_list = test_idx.tolist()
            result.append([arr[i] for i in train_list])
            result.append([arr[i] for i in test_list])
    return result


class KNeighborsRegressorManual:
    def __init__(self, n_neighbors=5, metric='manhattan'):
        self.n_neighbors = n_neighbors
        self.metric = metric
        self.X_train = None
        self.y_train = None

    def _manhattan_distance(self, x1, x2):
        # L1 расстояние: сумма модулей разностей
        return np.sum(np.abs(x1 - x2))

    def fit(self, X, y):
        # Просто запоминаем данные (KNN - ленивый алгоритм)
        self.X_train = np.array(X, dtype=np.float32)
        self.y_train = np.array(y, dtype=np.float32)
        return self

    def predict(self, X):
        X = np.array(X, dtype=np.float32)
        predictions = []

        # Для каждого тестового объекта ищем соседей
        for x in X:
            distances = []
            # Считаем расстояние до каждого объекта в тренировочном сете
            for i in range(len(self.X_train)):
                if self.metric == 'manhattan':
                    dist = self._manhattan_distance(x, self.X_train[i])
                else:
                    # Евклидово расстояние как запасной вариант
                    dist = np.sqrt(np.sum((x - self.X_train[i]) ** 2))
                distances.append((dist, self.y_train[i]))

            # Сортируем по расстоянию и берем k ближайших
            distances.sort(key=lambda item: item[0])
            k_nearest = distances[:self.n_neighbors]

            # Предсказание - среднее арифметическое целевых значений соседей
            pred = np.mean([y_val for _, y_val in k_nearest])
            predictions.append(pred)

        return np.array(predictions)


# Константы
PATCH_SIZE = 16
SAMPLES_DIR = "./generated_samples_manual_check"


# Класс для бинаризованного изображения
# Реализует преобразование изображения в строку "0101...",
# магические методы сравнения и извлечение признаков
class BinarizedImage:
    def __init__(self, image: np.ndarray, predicted_count: int = -1, true_count: int = -1):
        self._image = image
        self._predicted_count = predicted_count
        self._true_count = true_count
        self._binary = self._binarize()

    def _binarize(self) -> np.ndarray:
        # Конвертация в uint8 если изображение в формате float [0, 1]
        img = self._image
        if img.dtype in (np.float32, np.float64):
            img = (img * 255).astype(np.uint8)

        # Конвертация в оттенки серого если изображение цветное
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
        H, W = gray.shape
        ps = PATCH_SIZE
        rows, cols = H // ps, W // ps

        # Инициализация бинарного вектора
        binary = np.zeros(rows * cols, dtype=np.uint8)

        # Обход всех патчей
        idx = 0
        for r in range(rows):
            for c in range(cols):
                patch = gray[r * ps:(r + 1) * ps, c * ps:(c + 1) * ps]
                # Порог для автоматической бинаризации патча
                _, bpatch = cv2.threshold(patch, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                # Патч считается "чёрным" (содержит клетку) если средняя яркость < 127
                binary[idx] = 1 if bpatch.mean() < 127 else 0
                idx += 1
        return binary

    def __str__(self) -> str:
        # Возврат строкового представления бинарного вектора
        return ''.join(str(v) for v in self._binary)

    def __repr__(self) -> str:
        return (f'BinarizedImage(black_patches={self.black_patches_count}, '
                f'predicted={self._predicted_count}, true={self._true_count})')

    # Магические методы сравнения по предсказанному количеству клеток
    def __gt__(self, other: 'BinarizedImage') -> bool:
        return self._predicted_count > other._predicted_count

    def __lt__(self, other: 'BinarizedImage') -> bool:
        return self._predicted_count < other._predicted_count

    def __eq__(self, other: 'BinarizedImage') -> bool:
        return self._predicted_count == other._predicted_count

    def __ge__(self, other: 'BinarizedImage') -> bool:
        return self._predicted_count >= other._predicted_count

    def __le__(self, other: 'BinarizedImage') -> bool:
        return self._predicted_count <= other._predicted_count

    @property
    def features(self) -> np.ndarray:
        # Возврат признаков в формате, пригодном для sklearn (или нашего аналога)
        return self._binary.astype(np.float32)

    @property
    def black_patches_count(self) -> int:
        # Количество патчей, помеченных как "чёрные" (вероятно, содержат клетку)
        return int(self._binary.sum())


# Вспомогательные функции
def img_tensor_to_np(t: torch.Tensor) -> np.ndarray:
    # Конвертация torch-тензора в numpy-изображение [0, 255], uint8
    if t.max() <= 1.0:
        return (t.numpy() * 255).astype(np.uint8)
    return t.numpy().astype(np.uint8)


def parse_count_from_filename(filename: str) -> int:
    # Извлечение количества клеток из имени файла формата sample_XXXX_count_N.png
    match = re.search(r'count_(\d+)', filename)
    if match:
        return int(match.group(1))
    return -1


def load_images_from_folder(folder_path: str):
    # Загрузка изображений и меток из папки
    images_np = []
    labels = []
    filenames = sorted([f for f in os.listdir(folder_path) if f.endswith('.png')])

    for fname in filenames:
        fpath = os.path.join(folder_path, fname)
        img = cv2.imread(fpath)
        if img is None:
            continue
        # Конвертация BGR -> RGB для согласованности
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        count = parse_count_from_filename(fname)
        if count < 0:
            continue
        images_np.append(img)
        labels.append(count)

    return images_np, np.array(labels)


# Основная логика выполнения
if __name__ == '__main__':
    # Загрузка данных из папки с сгенерированными семплами
    print('Загрузка датасета из', SAMPLES_DIR)
    images_np, labels = load_images_from_folder(SAMPLES_DIR)

    if len(images_np) == 0:
        print('Изображения не найдены. Пожалуйста, сначала запустите генератор.')
        exit(1)

    print(f'Загружено изображений: {len(images_np)}, диапазон меток: [{labels.min()}, {labels.max()}]')

    # Извлечение признаков: бинаризация каждого изображения
    print('Извлечение признаков...')
    features = []
    for img in images_np:
        bi = BinarizedImage(img)
        features.append(bi.features)

    X = np.array(features)
    y = np.array(labels)
    print(f'Форма матрицы признаков: {X.shape}')

    # Разбиение на обучающую и тестовую выборки (ИСПОЛЬЗУЕМ НАШУ ФУНКЦИЮ)
    X_train, X_test, y_train, y_test, imgs_train, imgs_test = manual_train_test_split(
        X, y, images_np, test_size=0.25, random_state=42
    )

    # Обучение KNN-регрессора (ИСПОЛЬЗУЕМ НАШ КЛАСС)
    print('Обучение KNN-регрессора...')
    # n_neighbors=5, metric='manhattan' как в оригинале
    knn = KNeighborsRegressorManual(n_neighbors=5, metric='manhattan')
    knn.fit(X_train, y_train)

    # Предсказание на тестовой выборке
    y_pred_raw = knn.predict(X_test)
    y_pred = np.round(y_pred_raw).astype(int)

    # Расчёт MSE-метрик
    errors_sq = (y_pred - y_test) ** 2
    print('\nМетрики MSE:')
    print(f'  Среднее MSE: {errors_sq.mean():.4f}')
    print(f'  Мин. MSE: {errors_sq.min():.4f}')
    print(f'  Макс. MSE: {errors_sq.max():.4f}')

    # Создание объектов BinarizedImage для тестовых изображений
    bi_objects = []
    for img, pred in zip(imgs_test, y_pred):
        bi = BinarizedImage(img, predicted_count=int(pred))
        bi_objects.append(bi)

    # Демонстрация магических методов сравнения
    print('\n=== Демонстрация сравнения (магические методы) ===')
    if len(bi_objects) >= 2:
        bi1 = bi_objects[0]
        bi2 = bi_objects[1]
        print(f'Изобр. 1, бинарное (первые 40): {str(bi1)[:40]}...')
        print(f'Изобр. 2, бинарное (первые 40): {str(bi2)[:40]}...')
        print(f'Изобр. 1 > Изобр. 2: {bi1 > bi2}  '
              f'(предсказание: {bi1._predicted_count} против {bi2._predicted_count})')
        print(f'Изобр. 1 < Изобр. 2: {bi1 < bi2}')
        print(f'Изобр. 1 == Изобр. 2: {bi1 == bi2}')

    # Анализ пункта (e): сравнение количества чёрных патчей с истинным количеством клеток
    print('\n=== Пункт (e): чёрные патчи против истинного количества ===')
    correct_mask = y_pred == y_test
    correct_indices = np.where(correct_mask)[0]
    print(f'Верно предсказано: {len(correct_indices)} / {len(y_test)}')

    if len(correct_indices) > 0:
        mismatch_count = 0
        for idx in correct_indices:
            bi = bi_objects[idx]
            true_count = y_test[idx]
            if bi.black_patches_count != true_count:
                mismatch_count += 1
        pct = 100.0 * mismatch_count / len(correct_indices)
        print(f'Среди верно предсказанных изображений:')
        print(f'  black_patches_count != true_count: '
              f'{mismatch_count}/{len(correct_indices)} = {pct:.1f}%')
    else:
        print('Верно предсказанные образцы не найдены.')

Загрузка датасета из ./generated_samples_manual_check
Загружено изображений: 4724, диапазон меток: [0, 8]
Извлечение признаков...
Форма матрицы признаков: (4724, 256)
Обучение KNN-регрессора...

Метрики MSE:
  Среднее MSE: 9.0804
  Мин. MSE: 0.0000
  Макс. MSE: 64.0000

=== Демонстрация сравнения (магические методы) ===
Изобр. 1, бинарное (первые 40): 1111101001100000011111010100101001101000...
Изобр. 2, бинарное (первые 40): 0001100000100000010010000000010100110101...
Изобр. 1 > Изобр. 2: False  (предсказание: 4 против 4)
Изобр. 1 < Изобр. 2: False
Изобр. 1 == Изобр. 2: True

=== Пункт (e): чёрные патчи против истинного количества ===
Верно предсказано: 144 / 1181
Среди верно предсказанных изображений:
  black_patches_count != true_count: 139/144 = 96.5%
